# This notebook extracts the single-token Hindi words from the sarvamai/sarvam-2b-v0.5 model's vocubalary
> As for now, this notebook is not directly usable because we are using the translations from the english data that was already used in earlier experiments

# Environment Setup and Imports

In [3]:
import unicodedata
import re
import pandas as pd
from transformers import AutoTokenizer
from tqdm import tqdm

MODEL_ID = "sarvamai/sarvam-2b-v0.5" 
OUTPUT_CSV = "hindi_single_tokens.csv"

# Load Model Tokenizer & Inspect Vocabulary Size

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
vocab = tokenizer.get_vocab()
print(f"Loaded tokenizer from {MODEL_ID}")
print(f"Total vocabulary size: {len(vocab)}")

Loaded tokenizer from sarvamai/sarvam-2b-v0.5
Total vocabulary size: 64128


# Devanagari Filtering & Linguistic Sanitation Rules
This helper filters out subwords and fragments (like Maatra), keeping only valid full words

In [5]:
SPIECE_SPACE = "\u2581"

# Devanagari Unicode range: U+0900 to U+097F
DEVANAGARI_START = 0x0900
DEVANAGARI_END = 0x097F

# Unicode codepoint categories for dependent signs that cannot start a word
DEPENDENT_SIGNS = {
    0x0901, 0x0902, 0x0903,  # Chandrabindu, Anusvara, Visarga
    0x093C,                  # Nukta
    0x093E, 0x093F, 0x0940, 0x0941, 0x0942, 0x0943, 0x0944, # Dependent vowel signs (matras)
    0x0947, 0x0948, 0x094B, 0x094C, 0x094D,                  # Matras + Virama (Halant)
    0x0951, 0x0952, 0x0953, 0x0954,                          # Vedic stress accents
    0x0962, 0x0963                                           # Vowel signs
}

def clean_token_text(raw_tok: str) -> str:
    """Strips leading SentencePiece / Byte-level spaces and whitespace."""
    tok = raw_tok.replace(SPIECE_SPACE, "")
    tok = tok.strip()
    return tok

def is_devanagari_word(text: str) -> bool:
    """
    Checks if a string consists exclusively of Devanagari characters,
    is not empty, does not start with an unattached dependent diacritic/matra,
    and has a length > 1 (to eliminate single isolated alphabets/characters).
    """
    if not text or len(text) < 2:
        return False

    # Skip punctuation, digits, or non-Devanagari characters
    for ch in text:
        cp = ord(ch)
        if not (DEVANAGARI_START <= cp <= DEVANAGARI_END):
            return False

    # A standalone word cannot start with a dependent vowel sign, virama, or nukta
    if ord(text[0]) in DEPENDENT_SIGNS:
        return False
        
    # A standalone Hindi word typically does not end in a virama 
    if ord(text[-1]) == 0x094D:
        return False

    return True

# Scan Vocabulary and Extract Single-Token Hindi Words

In [6]:
single_token_hindi = []

for raw_token, token_id in tqdm(vocab.items(), desc="Scanning vocabulary"):
    # Discard special tokens or byte fallback representations like <unk>, <0x..>
    if raw_token.startswith("<") and raw_token.endswith(">"):
        continue

    word = clean_token_text(raw_token)
    
    if not is_devanagari_word(word):
        continue

    # Strict Dual Verification:
    # 1. Unspaced encoding: tokenizer.encode("शब्द", add_special_tokens=False)
    # 2. Spaced encoding:   tokenizer.encode(" शब्द", add_special_tokens=False)
    # A robust single token must tokenize to len == 1 in both contexts
    ids_unspaced = tokenizer.encode(word, add_special_tokens=False)
    ids_spaced = tokenizer.encode(" " + word, add_special_tokens=False)

    if len(ids_unspaced) == 1:
        single_token_hindi.append({
            "word": word,
            "raw_token": raw_token,
            "token_id": token_id,
            "unspaced_tok_id": ids_unspaced[0],
            "is_single_when_spaced": (len(ids_spaced) == 1),
            "spaced_tok_id": ids_spaced[0] if len(ids_spaced) == 1 else None,
            "char_length": len(word)
        })

df_hindi_tokens = pd.DataFrame(single_token_hindi)
# Remove duplicate surface words if both spaced and unspaced variants were captured
df_hindi_tokens = df_hindi_tokens.drop_duplicates(subset=["word"]).reset_index(drop=True)

print(f"Extracted {len(df_hindi_tokens)} unique single-token Hindi candidate words.")

Scanning vocabulary: 100%|██| 64128/64128 [00:01<00:00, 52367.61it/s]

Extracted 8386 unique single-token Hindi candidate words.


# Preview and Save the Curated List 

In [ ]:
# Sort by word length descending to inspect top complete words
df_hindi_tokens = df_hindi_tokens.sort_values(by="char_length", ascending=False).reset_index(drop=True)

print("Sample Single-Token Hindi Words:")
display(df_hindi_tokens[["word", "token_id", "char_length"]].head(20))

# Save the curated candidates
df_hindi_tokens.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Saved dataset to {OUTPUT_CSV}")

# Lexicon Validation & Noun Filtering
The paper specifically focused on concrete nouns (e.g., "flower", "cloud", "mountain") to minimize grammatical ambiguity. Cross-reference the extracted tokens against a standard Hindi lexicon (e.g., Hindi WordNet, or NLTK's Indian language corpus):

In [4]:
import pandas as pd
import nltk

# 2. Download the required Indian language corpus
nltk.download('indian', quiet=True)
from nltk.corpus import indian

# 3. Load the dataset you already generated
input_csv = "hindi_single_tokens.csv"
output_csv = "hindi_single_tokens_nouns.csv"

df_hindi_tokens = pd.read_csv(input_csv)

# 4. Extract known Hindi nouns from the NLTK POS-tagged corpus
print("Extracting noun lexicon from NLTK Indian corpus...")
hindi_tagged_words = indian.tagged_words('hindi.pos')

# Matches NN (Noun), NNP (Proper Noun), NNPC (Compound Proper Noun)
known_nouns = {w for w, pos in hindi_tagged_words if pos.startswith('NN')}

# 5. Apply the noun filter to your dataset
df_hindi_tokens['is_noun'] = df_hindi_tokens['word'].apply(lambda w: w in known_nouns)

# Create a new DataFrame containing only the verified nouns
df_hindi_nouns = df_hindi_tokens[df_hindi_tokens['is_noun']].copy().reset_index(drop=True)

print(f"Original single-token words: {len(df_hindi_tokens)}")
print(f"Confirmed single-token nouns: {len(df_hindi_nouns)}")

# 6. Preview and save the noun-filtered version
display(df_hindi_nouns[["word", "token_id", "char_length"]].head(15))

df_hindi_nouns.to_csv(output_csv, index=False, encoding="utf-8")
print(f"Saved noun-filtered dataset to {output_csv}")

Extracting noun lexicon from NLTK Indian corpus...
Original single-token words: 8386
Confirmed single-token nouns: 666


/raid/home/dgx1405/miniconda3/envs/venv/lib/python3.10/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/raid/home/dgx1405/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


,word,token_id,char_length
0,कार्यक्रमों,19810,11
1,यूनिवर्सिटी,28223,11
2,मुख्यमंत्री,11904,11
3,इंस्टीट्यूट,46368,11
4,श्रद्धांजलि,56817,11
5,ऑस्ट्रेलिया,9117,11
6,श्रद्धालुओं,58274,11
7,ऑस्ट्रेलियन,34948,11
8,अफगानिस्तान,61540,11
9,प्रतिक्रिया,13782,11


Saved noun-filtered dataset to hindi_single_tokens_nouns.csv
